# HW 4 
By Eugenia Tate for AIPI 530

## Problem 1: Campus help-desk staffing

Load packages 

In [1]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

Create Data

In [2]:
shiftData = pd.DataFrame(
    {"Shift": ["A", "B", "C", "D", "E"], "Cost": [120, 120, 120, 140, 210]}
)

demandData = pd.DataFrame({"Period": [1, 2, 3, 4], "Demand": [4, 6, 5, 3]})

coverageData = pd.DataFrame(
    {
        "A": [1, 1, 0, 0],
        "B": [0, 1, 1, 0],
        "C": [0, 0, 1, 1],
        "D": [1, 0, 0, 1],
        "E": [1, 1, 1, 1],
    },
    index=[1, 2, 3, 4],
)

coverageData.index.name = "Period"

paramData = pd.DataFrame({"maxE": [2]})

print(shiftData, demandData, coverageData, paramData, sep="\n\n")

  Shift  Cost
0     A   120
1     B   120
2     C   120
3     D   140
4     E   210

   Period  Demand
0       1       4
1       2       6
2       3       5
3       4       3

        A  B  C  D  E
Period               
1       1  0  0  1  1
2       1  1  0  0  1
3       0  1  1  0  1
4       0  0  1  1  1

   maxE
0     2


Define Model

In [3]:
m = AbstractModel()

# define sets
m.SHIFTS = Set()
m.PERIODS = Set()

# define parameters
m.cost = Param(m.SHIFTS)
m.demand = Param(m.PERIODS)
m.coverage = Param(m.PERIODS, m.SHIFTS)
m.maxE = Param()

# define decision variable
m.NShift = Var(m.SHIFTS, domain=NonNegativeIntegers)

# define implicit variable
m.NWorking = Var(m.PERIODS, domain=NonNegativeReals)

# define period staffing
def NWorking_Rule(m, t):
    return m.NWorking[t] == sum(m.coverage[t, s] * m.NShift[s] for s in m.SHIFTS)

m.Define_NWorking = Constraint(m.PERIODS, rule=NWorking_Rule)


# meet staffing requirements
def Demand_Rule(m, t):
    return m.NWorking[t] >= m.demand[t]

m.Cover_Demand = Constraint(m.PERIODS, rule=Demand_Rule)


# limit pattern availability
def Availability_Rule(m):
    return m.NShift["E"] <= m.maxE

m.Limit_E = Constraint(rule=Availability_Rule)


# minimize total payroll
def Min_Payroll_Rule(m):
    return sum(m.cost[s] * m.NShift[s] for s in m.SHIFTS)

m.Min_Payroll = Objective(rule=Min_Payroll_Rule, sense=minimize)

### Create instance

In [4]:
instanceData = {
    None: {
        "SHIFTS": {None: shiftData["Shift"].tolist()},
        "PERIODS": {None: demandData["Period"].tolist()},
        "cost": shiftData.set_index("Shift")["Cost"].to_dict(),
        "demand": demandData.set_index("Period")["Demand"].to_dict(),
        "coverage": coverageData.stack().to_dict(),
        "maxE": {None: paramData.loc[0, "maxE"]},
    }
}

instance = m.create_instance(instanceData)

Solve

In [5]:
solver = SolverFactory("glpk")
solution = solver.solve(instance)

assert check_optimal_termination(solution), "Solver did not terminate optimally."

Print Solution

In [6]:
print(solution["Solver"])

output = pd.DataFrame(
    [{"Shift": s, "Employees": value(instance.NShift[s])} for s in instance.SHIFTS]
)

coverage = pd.DataFrame(
    [
        {
            "Period": t,
            "Required": value(instance.demand[t]),
            "Working": value(instance.NWorking[t]),
            "Excess": (value(instance.NWorking[t]) - value(instance.demand[t])),
        }
        for t in instance.PERIODS
    ]
)

print(output.round(4).to_string(index=False))
print(coverage.round(4).to_string(index=False))
print(f"Total Payroll: ${value(instance.Min_Payroll):,.2f}")


- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 1
      Number of created subproblems: 1
  Error rc: 0
  Time: 0.03267407417297363

Shift  Employees
    A        2.0
    B        2.0
    C        1.0
    D        0.0
    E        2.0
 Period  Required  Working  Excess
      1         4      4.0     0.0
      2         6      6.0     0.0
      3         5      5.0     0.0
      4         3      3.0     0.0
Total Payroll: $1,020.00


**Explanation of results**

Assign 2 employees to A, 2 to B, 1 to C, none to D, and 2 to E.
The minimum total payroll is $1,020.

Coverage is 4, 6, 5, and 3 employees across the four periods, so
excess coverage is zero throughout. All four staffing requirements
and the two-employee availability limit for E are binding.

GLPK reports an optimal solution, certifying the global minimum.

## Problem 2: Nutrition Blend

Create data 

In [7]:
ingredientData = pd.DataFrame(
    {
        "Ingredient": ["Oats", "Soy", "Dried fruit"],
        "Cost": [2, 4, 3],
        "Protein": [10, 40, 5],
        "Fiber": [10, 5, 15],
        "Sugar": [2, 3, 30],
    }
)

paramData = pd.DataFrame(
    {
        "quantity": [1],
        "minimumProtein": [20],
        "minimumFiber": [8],
        "maximumSugar": [10],
        "maximumSoy": [0.60],
    }
)

print(ingredientData, paramData, sep="\n\n")

    Ingredient  Cost  Protein  Fiber  Sugar
0         Oats     2       10     10      2
1          Soy     4       40      5      3
2  Dried fruit     3        5     15     30

   quantity  minimumProtein  minimumFiber  maximumSugar  maximumSoy
0         1              20             8            10         0.6


Define Model

In [8]:
m = AbstractModel()

# define sets
m.INGREDIENTS = Set()

# define parameters
m.cost = Param(m.INGREDIENTS)
m.protein = Param(m.INGREDIENTS)
m.fiber = Param(m.INGREDIENTS)
m.sugar = Param(m.INGREDIENTS)

m.quantity = Param()
m.minimumProtein = Param()
m.minimumFiber = Param()
m.maximumSugar = Param()
m.maximumSoy = Param()

# define decision variable
m.x = Var(m.INGREDIENTS, domain=NonNegativeReals)

# define implicit variables
m.TotalCost = Var(domain=NonNegativeReals)
m.TotalProtein = Var(domain=NonNegativeReals)
m.TotalFiber = Var(domain=NonNegativeReals)
m.TotalSugar = Var(domain=NonNegativeReals)


# calculate total cost
def TotalCost_Rule(m):
    return m.TotalCost == sum(m.cost[i] * m.x[i] for i in m.INGREDIENTS)

m.Define_TotalCost = Constraint(rule=TotalCost_Rule)


# calculate total protein
def TotalProtein_Rule(m):
    return m.TotalProtein == sum(m.protein[i] * m.x[i] for i in m.INGREDIENTS)

m.Define_TotalProtein = Constraint(rule=TotalProtein_Rule)


# calculate total fiber
def TotalFiber_Rule(m):
    return m.TotalFiber == sum(m.fiber[i] * m.x[i] for i in m.INGREDIENTS)

m.Define_TotalFiber = Constraint(rule=TotalFiber_Rule)


# calculate total sugar
def TotalSugar_Rule(m):
    return m.TotalSugar == sum(m.sugar[i] * m.x[i] for i in m.INGREDIENTS)

m.Define_TotalSugar = Constraint(rule=TotalSugar_Rule)


# exact blend quantity
def Quantity_Rule(m):
    return sum(m.x[i] for i in m.INGREDIENTS) == m.quantity

m.Blend_Quantity = Constraint(rule=Quantity_Rule)

# minimum protein requirement
def Minimum_Protein_Rule(m):
    return m.TotalProtein >= m.minimumProtein

m.Minimum_Protein = Constraint(rule=Minimum_Protein_Rule)


# minimum fiber requirement
def Minimum_Fiber_Rule(m):
    return m.TotalFiber >= m.minimumFiber

m.Minimum_Fiber = Constraint(rule=Minimum_Fiber_Rule)


# maximum sugar limit
def Maximum_Sugar_Rule(m):
    return m.TotalSugar <= m.maximumSugar

m.Maximum_Sugar = Constraint(rule=Maximum_Sugar_Rule)


# maximum soy quantity
def Maximum_Soy_Rule(m):
    return m.x["Soy"] <= m.maximumSoy

m.Maximum_Soy = Constraint(rule=Maximum_Soy_Rule)


# minimize ingredient cost
def Min_Cost_Rule(m):
    return m.TotalCost

m.Min_Cost = Objective(rule=Min_Cost_Rule, sense=minimize)

Create Instance

In [9]:
instanceData = {
    None: {
        "INGREDIENTS": {None: ingredientData["Ingredient"].tolist()},
        "cost": ingredientData.set_index("Ingredient")["Cost"].to_dict(),
        "protein": ingredientData.set_index("Ingredient")["Protein"].to_dict(),
        "fiber": ingredientData.set_index("Ingredient")["Fiber"].to_dict(),
        "sugar": ingredientData.set_index("Ingredient")["Sugar"].to_dict(),
        "quantity": {None: paramData.loc[0, "quantity"]},
        "minimumProtein": {None: paramData.loc[0, "minimumProtein"]},
        "minimumFiber": {None: paramData.loc[0, "minimumFiber"]},
        "maximumSugar": {None: paramData.loc[0, "maximumSugar"]},
        "maximumSoy": {None: paramData.loc[0, "maximumSoy"]},
    }
}

instance = m.create_instance(instanceData)

Solve

In [10]:
solver = SolverFactory("glpk")
solution = solver.solve(instance)

assert check_optimal_termination(solution), "Solver did not terminate optimally."

Print Solution

In [11]:
print(solution["Solver"])

output = pd.DataFrame(
    [{"Ingredient": i, "Kilograms": value(instance.x[i])} for i in instance.INGREDIENTS]
)

print(output.round(4).to_string(index=False))
print(
    f"Total Quantity: {sum(value(instance.x[i]) for i in instance.INGREDIENTS):.4f} kg"
)
print(f"Total Cost: ${value(instance.TotalCost):.4f}")
print(f"Total Protein: {value(instance.TotalProtein):.4f} g")
print(f"Total Fiber: {value(instance.TotalFiber):.4f} g")
print(f"Total Sugar: {value(instance.TotalSugar):.4f} g")


- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 0
      Number of created subproblems: 0
  Error rc: 0
  Time: 0.058991193771362305

 Ingredient  Kilograms
       Oats     0.6667
        Soy     0.3333
Dried fruit     0.0000
Total Quantity: 1.0000 kg
Total Cost: $2.6667
Total Protein: 20.0000 g
Total Fiber: 8.3333 g
Total Sugar: 2.3333 g


**Explanation of results**

Use about 0.6667 kg of oats and 0.3333 kg of soy, with no dried fruit.
The 1-kg blend costs about $2.6667 and contains 20 g protein,
8.3333 g fiber, and 2.3333 g sugar.

The quantity and minimum-protein constraints are binding. Fiber
exceeds its minimum, while sugar and soy remain below their limits.
The nonnegativity bound for dried fruit is also binding.

GLPK reports an optimal solution, certifying the global minimum.